1. Importar librerías

In [2]:
import pandas as pd         # Manipulación y análisis de datos para el CVS
import numpy as np          # Operaciones numéricas y manejo de valores nulos
import plotly.express as px # Gráficos interactivos y visualizaciones dinámicas
import seaborn as sns       # Gráficos estadísticos
import matplotlib.pyplot as plt # Gráficos básicos 


2. Cargar archivo para visualización de los datos 

In [3]:
df = pd.read_csv("Violencia_Intrafamiliar_Policía_Nacional.csv")
df.head()

,DEPARTAMENTO,MUNICIPIO,CODIGO DANE,ARMAS MEDIOS,FECHA HECHO,GENERO,GRUPO ETARIO,CANTIDAD
0,CALDAS,Manizales (CT),17001000,SIN EMPLEO DE ARMAS,10/03/2026,FEMENINO,ADULTOS,2
1,ATLÁNTICO,Malambo,8433000,SIN EMPLEO DE ARMAS,15/03/2026,FEMENINO,ADULTOS,1
2,ATLÁNTICO,Luruaco,8421000,SIN EMPLEO DE ARMAS,09/03/2026,FEMENINO,ADULTOS,1
3,HUILA,Pitalito,41551000,SIN EMPLEO DE ARMAS,14/03/2026,FEMENINO,ADULTOS,1
4,CUNDINAMARCA,Anapoima,25035000,SIN EMPLEO DE ARMAS,28/01/2026,FEMENINO,ADULTOS,1


OBSERVACIÓN:
No se va a trabajar con la columna CODIGO DANE, pues no proporciona un valor significativo para nuestro análisis, pero antes de limpiar los datos esta columna nos ayudo a determinar que dentro de nuestro dataframe no hay valores duplicados.

In [4]:
cols = [
    "DEPARTAMENTO",
    "MUNICIPIO",
    "ARMAS MEDIOS",
    "FECHA HECHO",
    "GENERO",
    "GRUPO ETARIO",
    "CANTIDAD",
    "CODIGO DANE"
]

df[df.duplicated(subset=cols, keep=False)].sort_values(cols)

,DEPARTAMENTO,MUNICIPIO,CODIGO DANE,ARMAS MEDIOS,FECHA HECHO,GENERO,GRUPO ETARIO,CANTIDAD


3. Revisión de los nombres de las columnas 

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 682558 entries, 0 to 682557
Data columns (total 8 columns):
 #   Column        Non-Null Count   Dtype
---  ------        --------------   -----
 0   DEPARTAMENTO  682558 non-null  str  
 1   MUNICIPIO     682554 non-null  str  
 2   CODIGO DANE   682558 non-null  int64
 3   ARMAS MEDIOS  682558 non-null  str  
 4   FECHA HECHO   682558 non-null  str  
 5   GENERO        679671 non-null  str  
 6   GRUPO ETARIO  681416 non-null  str  
 7   CANTIDAD      682558 non-null  int64
dtypes: int64(2), str(6)
memory usage: 41.7 MB


4. Copia del csv y elimanción columna Código Dane

In [6]:
data = df.drop(columns=['CODIGO DANE'])

5. Cambia del tipo de dato para la columna fecha hecho 

In [7]:
data["FECHA HECHO"] = pd.to_datetime(data["FECHA HECHO"], format="%d/%m/%Y")
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 682558 entries, 0 to 682557
Data columns (total 7 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   DEPARTAMENTO  682558 non-null  str           
 1   MUNICIPIO     682554 non-null  str           
 2   ARMAS MEDIOS  682558 non-null  str           
 3   FECHA HECHO   682558 non-null  datetime64[us]
 4   GENERO        679671 non-null  str           
 5   GRUPO ETARIO  681416 non-null  str           
 6   CANTIDAD      682558 non-null  int64         
dtypes: datetime64[us](1), int64(1), str(5)
memory usage: 36.5 MB


6. Ajustar nombres de las columnas

In [8]:
data.rename(columns={'ARMAS MEDIOS': 'ARMAS_MEDIOS'}, inplace=True)
data.rename(columns={'FECHA HECHO': 'FECHA_HECHO'}, inplace=True)
data.rename(columns={'GRUPO ETARIO': 'GRUPO_ETARIO'}, inplace=True)

7. Revisión datos nulos o vacíos 

In [9]:
# Verificación de la cantidad de valores nulos por columna
data.isnull().sum()

# Verificación de campos vacíos
(data == "").sum()

DEPARTAMENTO    0
MUNICIPIO       0
ARMAS_MEDIOS    0
FECHA_HECHO     0
GENERO          0
GRUPO_ETARIO    0
CANTIDAD        0
dtype: int64

8. Verifación de datos naN

In [10]:
#Verificación la cantidad de valores nulos (NaN) presentes en cada columna
data.isna().sum()

DEPARTAMENTO       0
MUNICIPIO          4
ARMAS_MEDIOS       0
FECHA_HECHO        0
GENERO          2887
GRUPO_ETARIO    1142
CANTIDAD           0
dtype: int64

9. Reemplazo de datos naN por la palabra desconocido y no especificado

In [11]:
data["MUNICIPIO"] = data["MUNICIPIO"].fillna("DESCONOCIDO") #Fillna reempleza naN por desconocido o no especificado
data["GENERO"] = data["GENERO"].fillna("NO ESPECIFICADO")
data["GRUPO_ETARIO"] = data["GRUPO_ETARIO"].fillna("NO ESPECIFICADO")

10. Estandarización de caracteres: Estilo de escritura y quitar tildes

In [12]:
data["MUNICIPIO"] = data["MUNICIPIO"].str.upper()

data['MUNICIPIO'] = data['MUNICIPIO'].str.normalize('NFKD') \
    .str.encode('ascii', errors='ignore') \
    .str.decode('utf-8')
    
data['DEPARTAMENTO'] = data['DEPARTAMENTO'].str.normalize('NFKD') \
    .str.encode('ascii', errors='ignore') \
    .str.decode('utf-8')

GRÁFICAS

1. GRÁFICA: La gráfica TOP 10 MUNICIPIOS - VIOLENCIA EN MUJERES, muestra los diez municipios con mayor número de casos de violencia contra la mujer, permitiendo identificar cuáles son las zonas donde este problema es más frecuente.

In [22]:
# Normalizar género
data['GENERO'] = data['GENERO'].str.upper()

# Filtrar femenino
femenino = data[data['GENERO'] == 'FEMENINO']

# Agrupar por municipio
violencia = femenino.groupby('MUNICIPIO')['CANTIDAD'].sum().reset_index()

# Top 10
violencia = violencia.sort_values(by='CANTIDAD', ascending=False).head(10)

# Gráfica
fig = px.bar(
    violencia,
    x='MUNICIPIO',
    y='CANTIDAD',
    text='CANTIDAD',
    title='TOP 10 MUNICIPIOS - VIOLENCIA EN MUJERES'
)

# ESTILO DEL GRÁFICO

fig.update_layout(
    title=dict(
        text='TOP 10 MUNICIPIOS - VIOLENCIA EN MUJERES',
        x=0.5,  # centra el título
        xanchor='center',
        font=dict(size=20)
    ),
    
    xaxis=dict(
        tickangle=0,  # sin inclinación
        tickfont=dict(size=7)  # municipios más pequeños
    )
)

fig.show()